# Pipeline chọn cột

## Các bước trong pipeline:
1. Lấy ra những bảng liên quan để trả lời câu hỏi
2. Render user prompt bằng bảng liên quan, user prompt chứa bảng, cột, mô tả cột và quan hệ
3. Đưa user prompt và system prompt vào LLM
4. Kiểm tra những bảng trả về của LLM (chỉ được chứa những cột trong schema)
5. Thêm những cột join trong trường hợp kết quả thiếu những cột join cần thiết
6. Output kết quả

In [2]:
#===== SETUP =====

import sys
import os
import json

sys.path.insert(0, os.path.abspath('.'))

from dotenv import load_dotenv
load_dotenv('pipeline/.env')

from pipeline.get_columns import (
    load_schema_file,
    load_column_descriptions,
    parse_all_tables,
    get_tables_with_columns,
    load_column_template,
    render_column_template,
    call_llm,
    validate_columns,
    add_missing_join_columns
)
from pipeline.get_final import get_final_tables
from pipeline.instructions.system_prompt_column_en import SYSTEM_INSTRUCTION as SYSTEM_COL_EN


c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using gpt-oss-20b for table selection
USE_INSTRUCTION: true


In [3]:
#===== GET RELEVANT TABLE =====
query = "Doanh thu theo danh mục sản phẩm và thương hiệu của từng khách hàng"

table_result = get_final_tables(query, top_k=10)

final_tables = table_result.get("final_tables", [])

print(f"Selected Tables: {final_tables}")

Selected Tables: ['don_hang', 'chi_tiet_don_hang', 'bien_the_san_pham', 'san_pham', 'khach_hang', 'danh_muc_san_pham']


In [4]:
#==== LOAD SCHEMA ====
schema_text = load_schema_file()
all_tables = parse_all_tables(schema_text)

print(f"Total tables in schema: {len(all_tables)}")
print(f"\nExample - 'don_hang' table structure:")
print(f"  Description: {all_tables['don_hang']['description']}")
print(f"  Columns: {[c['name'] for c in all_tables['don_hang']['columns']]}")

Total tables in schema: 20

Example - 'don_hang' table structure:
  Description: Đơn hàng của khách hàng; tổng tiền được tính từ chi_tiet_don_hang + phí/giảm giá.
  Columns: ['don_hang_id', 'khach_hang_id', 'dia_chi_giao_id', 'trang_thai', 'tong_tien_hang', 'phi_van_chuyen', 'giam_gia', 'thue', 'tong_thanh_toan', 'tien_te', 'ngay_dat', 'ngay_cap_nhat']


In [5]:
#==== LOAD TABLE DESCRIPTION ====
column_descriptions = load_column_descriptions()

print(f"Tables with descriptions: {list(column_descriptions.keys())}")
print(f"\nExample - 'don_hang' column descriptions:")
for col, desc in column_descriptions.get('don_hang', {}).items():
    print(f"  {col}: {desc}")

Tables with descriptions: ['khach_hang', 'dia_chi_khach_hang', 'danh_muc_san_pham', 'thuong_hieu', 'nha_ban', 'san_pham', 'bien_the_san_pham', 'kho', 'ton_kho', 'don_hang', 'chi_tiet_don_hang', 'thanh_toan', 'khuyen_mai', 'ap_dung_khuyen_mai', 'van_chuyen', 'su_kien_van_don', 'tra_hang_hoan_tien', 'chi_tiet_tra_hang', 'danh_gia_san_pham', 'nhat_ky_he_thong']

Example - 'don_hang' column descriptions:
  don_hang_id: Định danh đơn hàng.
  khach_hang_id: Khách hàng đặt đơn.
  dia_chi_giao_id: Địa chỉ giao hàng.
  trang_thai: Trạng thái đơn hàng.
  tong_tien_hang: Tổng tiền hàng.
  phi_van_chuyen: Phí vận chuyển.
  giam_gia: Tổng giảm giá.
  thue: Thuế áp dụng.
  tong_thanh_toan: Tổng tiền phải thanh toán của khách hàng.
  tien_te: Đơn vị tiền tệ.
  ngay_dat: Thời điểm đặt hàng.
  ngay_cap_nhat: Thời điểm cập nhật đơn hàng.


In [6]:
#==== COMBINE ORIGINAL SCHEMA WITH DESCRIPTION ====
tables_with_columns = get_tables_with_columns(final_tables, all_tables, column_descriptions)

print(f"Prepared {len(tables_with_columns)} tables for LLM prompt:\n")
for table in tables_with_columns:
    print(f"Table: {table['name']}")
    print(f"  Description: {table['description']}")
    print(f"  Columns:")
    for col in table['columns'][:3]: 
        desc = col.get('description', '')
        print(f"    - {col['name']} ({col['type']}): {desc[:50]}..." if len(desc) > 50 else f"    - {col['name']} ({col['type']}): {desc}")
    if len(table['columns']) > 3:
        print(f"    ... and {len(table['columns']) - 3} more columns")
    print()

Prepared 6 tables for LLM prompt:

Table: don_hang
  Description: Đơn hàng của khách hàng; tổng tiền được tính từ chi_tiet_don_hang + phí/giảm giá.
  Columns:
    - don_hang_id (UUID): Định danh đơn hàng.
    - khach_hang_id (UUID): Khách hàng đặt đơn.
    - dia_chi_giao_id (UUID): Địa chỉ giao hàng.
    ... and 9 more columns

Table: chi_tiet_don_hang
  Description: Dòng hàng (line item) trong đơn; mỗi SKU có thể xuất hiện nhiều lần ở các đơn khác nhau.
  Columns:
    - chi_tiet_id (UUID): Định danh chi tiết đơn hàng.
    - don_hang_id (UUID): Đơn hàng chứa dòng.
    - sku_id (UUID): Biến thể được mua.
    ... and 4 more columns

Table: bien_the_san_pham
  Description: Biến thể sản phẩm (SKU) theo màu/size…; giá và mã SKU quản lý ở đây.
  Columns:
    - sku_id (UUID): Định danh biến thể (SKU).
    - san_pham_id (UUID): Sản phẩm cha.
    - ma_sku (TEXT): Mã SKU (duy nhất).
    ... and 8 more columns

Table: san_pham
  Description: Sản phẩm “mức cha” (SPU) do nhà bán đăng; SKU nằm ở bản

In [7]:
# ===== RENDER USER PROMPT FROM TEMPLATE ====
template_text = load_column_template()
user_prompt = render_column_template(template_text, query, tables_with_columns)

print("Rendered Prompt (first 2000 chars):")
print("=" * 60)
print(user_prompt)
print("...")

Rendered Prompt (first 2000 chars):
                               YOUR TASK

You are a COLUMN SELECTION SYSTEM in the Text-to-SQL pipeline.

OBJECTIVE:
Select ONLY the MINIMUM columns required to answer the question.
Be STRICT - do not include extra columns.

CRITICAL - AVOID OVER-SELECTION:
- Select the FEWEST columns that can answer the question
- Do NOT include columns "just in case"
- Do NOT select all columns from a table
- If a column is not directly needed, do NOT include it

MANDATORY CONSTRAINTS:
- ONLY select columns from the tables provided below
- DO NOT create new column names
- DO NOT output SQL queries

COLUMN SELECTION RULES (BE STRICT):

1. JOIN columns: ONLY specific PK/FK columns needed to connect tables
   - Do NOT include extra ID columns that are not in the JOIN path

2. SELECT columns: ONLY columns needed for output
   - ONLY columns explicitly requested or needed for calculations
   - Do NOT include "related" columns that weren't asked for

3. WHERE columns: ON

In [8]:
# ==== CALL LLM WITH USER AND SYSTEM PROMPT
llm_result = call_llm(SYSTEM_COL_EN, user_prompt)

print("LLM Raw Response:")
print("=" * 60)
# Show the results without reasoning
display_result = {k: v for k, v in llm_result.items() if k not in ['reasoning_content', 'token_usage']}
print(json.dumps(display_result, indent=2, ensure_ascii=False))

LLM Raw Response:
{
  "results": [
    {
      "table_name": "don_hang",
      "table_reason": "Provides the link between orders and customers and connects to line items",
      "columns": [
        "don_hang_id",
        "khach_hang_id"
      ],
      "column_reasons": [
        "Join key to connect with chi_tiet_don_hang",
        "Identifies the customer for grouping revenue per customer"
      ]
    },
    {
      "table_name": "chi_tiet_don_hang",
      "table_reason": "Contains the monetary amount of each purchased SKU",
      "columns": [
        "don_hang_id",
        "sku_id",
        "thanh_tien"
      ],
      "column_reasons": [
        "Join key to connect with don_hang",
        "Join key to connect with bien_the_san_pham",
        "Revenue amount to be summed for each category‑brand‑customer combination"
      ]
    },
    {
      "table_name": "bien_the_san_pham",
      "table_reason": "Maps each SKU to its parent product (SPU)",
      "columns": [
        "sku_id",
   

In [9]:
# ===== VALIDATE COLUMN =====
# Build allowed columns map
allowed_columns = {}
for t in tables_with_columns:
    allowed_columns[t["name"]] = [c["name"] for c in t["columns"]]

print("Allowed columns per table:")
for table, cols in allowed_columns.items():
    print(f"  {table}: {cols[:5]}..." if len(cols) > 5 else f"  {table}: {cols}")

# Validate
validated_columns = validate_columns(llm_result, allowed_columns)

print("\nValidated columns:")
for table, cols in validated_columns.items():
    print(f"  {table}: {cols}")

Allowed columns per table:
  don_hang: ['don_hang_id', 'khach_hang_id', 'dia_chi_giao_id', 'trang_thai', 'tong_tien_hang']...
  chi_tiet_don_hang: ['chi_tiet_id', 'don_hang_id', 'sku_id', 'so_luong', 'don_gia']...
  bien_the_san_pham: ['sku_id', 'san_pham_id', 'ma_sku', 'mau_sac', 'kich_co']...
  san_pham: ['san_pham_id', 'nha_ban_id', 'danh_muc_id', 'thuong_hieu_id', 'ten_san_pham']...
  khach_hang: ['khach_hang_id', 'ho_ten', 'email', 'so_dien_thoai', 'ngay_sinh']...
  danh_muc_san_pham: ['danh_muc_id', 'danh_muc_cha_id', 'ten_danh_muc', 'mo_ta', 'trang_thai']...

Validated columns:
  don_hang: ['don_hang_id', 'khach_hang_id']
  chi_tiet_don_hang: ['don_hang_id', 'sku_id', 'thanh_tien']
  bien_the_san_pham: ['sku_id', 'san_pham_id']
  san_pham: ['san_pham_id', 'danh_muc_id', 'thuong_hieu_id']
  danh_muc_san_pham: ['danh_muc_id', 'ten_danh_muc']
  khach_hang: ['khach_hang_id', 'ho_ten']


In [10]:
# ==== ADD MISSING JOIN COLUMNS ====
final_columns = add_missing_join_columns(validated_columns, all_tables)

print("Before adding JOIN columns:")
for table, cols in validated_columns.items():
    print(f"  {table}: {cols}")

print("\nAfter adding JOIN columns:")
for table, cols in final_columns.items():
    print(f"  {table}: {cols}")

Before adding JOIN columns:
  don_hang: ['don_hang_id', 'khach_hang_id']
  chi_tiet_don_hang: ['don_hang_id', 'sku_id', 'thanh_tien']
  bien_the_san_pham: ['sku_id', 'san_pham_id']
  san_pham: ['san_pham_id', 'danh_muc_id', 'thuong_hieu_id']
  danh_muc_san_pham: ['danh_muc_id', 'ten_danh_muc']
  khach_hang: ['khach_hang_id', 'ho_ten']

After adding JOIN columns:
  don_hang: ['don_hang_id', 'khach_hang_id']
  chi_tiet_don_hang: ['don_hang_id', 'sku_id', 'thanh_tien']
  bien_the_san_pham: ['sku_id', 'san_pham_id']
  san_pham: ['san_pham_id', 'danh_muc_id', 'thuong_hieu_id']
  danh_muc_san_pham: ['danh_muc_id', 'ten_danh_muc', 'danh_muc_cha_id']
  khach_hang: ['khach_hang_id', 'ho_ten']


In [11]:
# ==== RESULT =====
print(f"\nTables: {list(final_columns.keys())}")
print(f"\nColumns by table:")
for table, cols in final_columns.items():
    print(f"  {table}:")
    for col in cols:
        print(f"    - {col}")

print(f"\nToken usage:")
token_usage = llm_result.get('token_usage', {})
print(f"  Prompt: {token_usage.get('prompt_tokens', 0):,}")
print(f"  Completion: {token_usage.get('completion_tokens', 0):,}")


Tables: ['don_hang', 'chi_tiet_don_hang', 'bien_the_san_pham', 'san_pham', 'danh_muc_san_pham', 'khach_hang']

Columns by table:
  don_hang:
    - don_hang_id
    - khach_hang_id
  chi_tiet_don_hang:
    - don_hang_id
    - sku_id
    - thanh_tien
  bien_the_san_pham:
    - sku_id
    - san_pham_id
  san_pham:
    - san_pham_id
    - danh_muc_id
    - thuong_hieu_id
  danh_muc_san_pham:
    - danh_muc_id
    - ten_danh_muc
    - danh_muc_cha_id
  khach_hang:
    - khach_hang_id
    - ho_ten

Token usage:
  Prompt: 2,626
  Completion: 1,243


## Độ chính xác pipeline chọn bảng

### Test set:

Mỗi test case chứa:
- Câu hỏi
- Những bảng **tối thiểu** cần sử dụng
- Với mỗi bảng, những cột **tối thiểu**  cần sử dụng 


### Tiêu chí đánh giá:
**Recall**: 
- Recall = Số cột chọn đúng / Số cột cần dùng
- Pipeline có chọn đủ bảng hay không?

**Precision**: 
- Precision = Số bảng LLM chọn đúng / Số bảng LLM chọn ra
- Những bảng được chọn có cần thiết hay không?

### LLM được sử dụng:
- **gpt-oss-20b**: Model nhỏ hơn, nhanh hơn
- **gpt-oss-120b**: Model lớn hơn, có khả năng reasoning tốt hơn


In [12]:
# ===== RUN EVALUATION WITH GPT-OSS-20B ====
import runpy

os.environ["COLUMN_TEST_OUT"] = "results/oss_20b_column_accuracy.txt"
os.environ["COLUMN_SELECTION_LLM_MODEL"] = "gpt-oss-20b"

print("Running column accuracy test with gpt-oss-20b...")
runpy.run_path("llm_column_accuracy.py", run_name="__main__")



Running column accuracy test with gpt-oss-20b...
Results written to: results/oss_20b_column_accuracy.txt
Average Recall:    0.8704
Average Precision: 0.7833


{'__name__': '__main__',
 '__doc__': '\nEvaluate column selection accuracy for the pipeline:\n  - Table selection (from previous step)\n  - LLM column selection\n\nMetrics:\n  - Recall: Are all required columns selected?\n  - Precision: Are selected columns relevant?\n\nOutputs:\n  - Per query breakdown\n  - Overall averages\n',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'llm_column_accuracy.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.

In [13]:
# ===== RUN EVALUATION WITH GPT-OSS-120B ====
import runpy

os.environ["COLUMN_TEST_OUT"] = "results/oss_120b_column_accuracy.txt"
os.environ["COLUMN_SELECTION_LLM_MODEL"] = "gpt-oss-120b"

print("Running column accuracy test with gpt-oss-120b...")
runpy.run_path("llm_column_accuracy.py", run_name="__main__")


Running column accuracy test with gpt-oss-120b...
Results written to: results/oss_120b_column_accuracy.txt
Average Recall:    0.8889
Average Precision: 0.8205


{'__name__': '__main__',
 '__doc__': '\nEvaluate column selection accuracy for the pipeline:\n  - Table selection (from previous step)\n  - LLM column selection\n\nMetrics:\n  - Recall: Are all required columns selected?\n  - Precision: Are selected columns relevant?\n\nOutputs:\n  - Per query breakdown\n  - Overall averages\n',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'llm_column_accuracy.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.

In [14]:
#===== READ GPT-OSS-20B RESULTS =====
from pathlib import Path

path = Path("results/oss_20b_column_accuracy.txt")
print(path.read_text(encoding="utf-8"))

TEST CASE #1
QUERY: Tổng doanh thu theo từng nhà bán trong quý trước
REQUIRED TABLES: nha_ban, san_pham, bien_the_san_pham, chi_tiet_don_hang, don_hang

REQUIRED COLUMNS:
  nha_ban: nha_ban_id, ten_cua_hang
  san_pham: san_pham_id, nha_ban_id
  bien_the_san_pham: sku_id, san_pham_id
  chi_tiet_don_hang: don_hang_id, sku_id, thanh_tien
  don_hang: don_hang_id, ngay_dat

SELECTED TABLES: don_hang, chi_tiet_don_hang, bien_the_san_pham, san_pham, nha_ban

SELECTED COLUMNS:
  don_hang: don_hang_id, ngay_dat
  chi_tiet_don_hang: don_hang_id, sku_id, thanh_tien
  bien_the_san_pham: sku_id, san_pham_id
  san_pham: san_pham_id, nha_ban_id
  nha_ban: nha_ban_id, ten_cua_hang

Column Recall:    1.000
Column Precision: 1.000

COLUMN COMPARISON
------------------------------------------------------------
TABLE: bien_the_san_pham
  [OK]     san_pham_id, sku_id

TABLE: chi_tiet_don_hang
  [OK]     don_hang_id, sku_id, thanh_tien

TABLE: don_hang
  [OK]     don_hang_id, ngay_dat

TABLE: nha_ban
  [OK]

In [15]:
#===== READ GPT-OSS-120B RESULTS =====
from pathlib import Path

path = Path("results/oss_120b_column_accuracy.txt")
print(path.read_text(encoding="utf-8"))

TEST CASE #1
QUERY: Tổng doanh thu theo từng nhà bán trong quý trước
REQUIRED TABLES: nha_ban, san_pham, bien_the_san_pham, chi_tiet_don_hang, don_hang

REQUIRED COLUMNS:
  nha_ban: nha_ban_id, ten_cua_hang
  san_pham: san_pham_id, nha_ban_id
  bien_the_san_pham: sku_id, san_pham_id
  chi_tiet_don_hang: don_hang_id, sku_id, thanh_tien
  don_hang: don_hang_id, ngay_dat

SELECTED TABLES: don_hang, chi_tiet_don_hang, bien_the_san_pham, san_pham, nha_ban

SELECTED COLUMNS:
  don_hang: don_hang_id, ngay_dat
  chi_tiet_don_hang: don_hang_id, sku_id, thanh_tien
  bien_the_san_pham: sku_id, san_pham_id
  san_pham: san_pham_id, nha_ban_id
  nha_ban: nha_ban_id, ten_cua_hang

Column Recall:    1.000
Column Precision: 1.000

COLUMN COMPARISON
------------------------------------------------------------
TABLE: bien_the_san_pham
  [OK]     san_pham_id, sku_id

TABLE: chi_tiet_don_hang
  [OK]     don_hang_id, sku_id, thanh_tien

TABLE: don_hang
  [OK]     don_hang_id, ngay_dat

TABLE: nha_ban
  [OK]